[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/optimization/04_line_search_newton_quasi_newton/first_principles.ipynb)

# Topic 04: Line Search, Newton & Quasi-Newton Methods

## 1. First-Principles Intuition & Motivation

An iterative optimizer repeats one primitive: from the current point $\mathbf{x}_k$, pick a direction $\mathbf{d}_k$ and a step length $\alpha_k$, then move to $\mathbf{x}_{k+1} = \mathbf{x}_k + \alpha_k \mathbf{d}_k$. Topic 03 fixed the direction to $-\nabla f(\mathbf{x}_k)$ and studied constant steps. This module relaxes both choices and asks the two questions that separate toy algorithms from production solvers:

1. **How far should we step?** A step that merely decreases $f$ is not enough — decreases can shrink geometrically toward a point that is not even stationary. We need *sufficient* decrease, quantified relative to the step length and the local slope.
2. **Can we choose better directions than steepest descent?** Steepest descent is greedy and myopic: it ignores curvature and zig-zags in ill-conditioned valleys. Using the Hessian — exactly (Newton) or approximately (quasi-Newton) — replaces the crawl with superlinear or quadratic convergence.

### The one-dimensional restriction

Every line search reasons about the scalar function

$$
\varphi(\alpha) = f(\mathbf{x}_k + \alpha \mathbf{d}_k), \qquad \varphi'(\alpha) = \nabla f(\mathbf{x}_k + \alpha \mathbf{d}_k)^T \mathbf{d}_k
$$

If $\mathbf{d}_k$ is a **descent direction**, meaning $\varphi'(0) = \nabla f(\mathbf{x}_k)^T \mathbf{d}_k \lt 0$, then small enough steps decrease $f$. The art of line search is to accept steps that are neither too long (overshooting the valley) nor too short (crawling), using only a few evaluations of $\varphi$.

The art of direction choice is to make $\varphi$ as easy as possible: Newton's direction is engineered so that a unit step $\alpha = 1$ jumps directly to the minimizer of the local quadratic model of $f$.

## 2. Rigorous Mathematical Definitions & Theorem Statements

**Definition 1 (Descent direction).** $\mathbf{d} \in \mathbb{R}^n$ is a descent direction for $f$ at $\mathbf{x}$ if $\nabla f(\mathbf{x})^T \mathbf{d} \lt 0$.

**Definition 2 (Armijo sufficient-decrease condition).** Fix $c_1 \in (0, 1)$. The step $\alpha \gt 0$ satisfies the Armijo condition along the descent direction $\mathbf{d}_k$ if

$$
f(\mathbf{x}_k + \alpha \mathbf{d}_k) \le f(\mathbf{x}_k) + c_1\, \alpha\, \nabla f(\mathbf{x}_k)^T \mathbf{d}_k
$$

The right-hand side is a line lying *below* the tangent line at $\alpha = 0$ (since $0 \lt c_1 \lt 1$ and the slope is negative): accepted steps must beat a fixed fraction of the first-order predicted decrease.

**Definition 3 (Wolfe conditions).** Fix $0 \lt c_1 \lt c_2 \lt 1$. The step $\alpha$ satisfies the Wolfe conditions if, in addition to the Armijo condition,

$$
\nabla f(\mathbf{x}_k + \alpha \mathbf{d}_k)^T \mathbf{d}_k \ge c_2\, \nabla f(\mathbf{x}_k)^T \mathbf{d}_k
$$

The curvature condition demands that the slope of $\varphi$ has risen from its initial value $\varphi'(0) \lt 0$ to at least $c_2 \varphi'(0)$ — the step must have consumed most of the available downhill slope, forbidding vanishingly small steps.

Standard parameter choices are $c_1 = 10^{-4}$ (a very permissive decrease bar) and $c_2 = 0.9$ for quasi-Newton directions or $c_2 = 0.1$ for nonlinear conjugate gradient.

**Definition 4 (Backtracking line search).** Given $\alpha_0 \gt 0$, contraction $\rho \in (0, 1)$, and $c_1 \in (0, 1)$: try $\alpha = \alpha_0, \rho\alpha_0, \rho^2\alpha_0, \dots$ and accept the first $\alpha$ satisfying the Armijo condition.

**Definition 5 (Newton direction).** At a point where $\nabla^2 f(\mathbf{x}_k) \succ 0$, the Newton direction is the minimizer of the local quadratic model, i.e. the solution of

$$
\nabla^2 f(\mathbf{x}_k)\, \mathbf{p}_k = -\nabla f(\mathbf{x}_k)
$$

**Definition 6 (Secant pair and secant equation).** With $\mathbf{s}_k = \mathbf{x}_{k+1} - \mathbf{x}_k$ and $\mathbf{y}_k = \nabla f(\mathbf{x}_{k+1}) - \nabla f(\mathbf{x}_k)$, a quasi-Newton approximation $B_{k+1}$ of the Hessian must satisfy the **secant equation**

$$
B_{k+1}\, \mathbf{s}_k = \mathbf{y}_k
$$

which forces the model to reproduce the observed change of the gradient along the actual step.

The secant equation is the multidimensional analogue of the one-dimensional secant slope $f''(x) \approx (f'(x_{k+1}) - f'(x_k))/(x_{k+1} - x_k)$: curvature is learned from two gradient evaluations rather than computed from second derivatives.

**Theorem 1 (Existence of Armijo steps).** If $f \in \mathcal{C}^1$, $\mathbf{d}_k$ is a descent direction, and $c_1 \in (0, 1)$, then there exists $\bar{\alpha} \gt 0$ such that the Armijo condition holds for all $\alpha \in (0, \bar{\alpha}]$. Consequently backtracking terminates after finitely many contractions.

**Theorem 2 (Existence of Wolfe steps).** If additionally $f$ is bounded below along the ray $\{\mathbf{x}_k + \alpha \mathbf{d}_k : \alpha \gt 0\}$ and $0 \lt c_1 \lt c_2 \lt 1$, then there exist nonempty intervals of steps satisfying both Wolfe conditions.

**Theorem 3 (Zoutendijk).** Suppose $f$ is bounded below, $\nabla f$ is $L$-Lipschitz on the sublevel set of $\mathbf{x}_0$, and every iteration takes a descent step satisfying the Wolfe conditions. Then

$$
\sum_{k=0}^{\infty} \cos^2\theta_k\, \lVert \nabla f(\mathbf{x}_k)\rVert^2 \lt \infty, \qquad \cos\theta_k = \frac{-\nabla f(\mathbf{x}_k)^T \mathbf{d}_k}{\lVert \nabla f(\mathbf{x}_k)\rVert\, \lVert \mathbf{d}_k\rVert}
$$

In particular, if the directions stay uniformly non-orthogonal to the negative gradient ($\cos\theta_k \ge \delta \gt 0$), then $\lVert \nabla f(\mathbf{x}_k)\rVert \to 0$.

**Theorem 4 (Newton local quadratic convergence).** Let $\nabla^2 f$ be $M$-Lipschitz near a minimizer $\mathbf{x}^{\ast}$ with $\nabla f(\mathbf{x}^{\ast}) = \mathbf{0}$ and $\nabla^2 f(\mathbf{x}^{\ast}) \succeq m I$, $m \gt 0$. Then for $\mathbf{x}_0$ close enough to $\mathbf{x}^{\ast}$, full Newton steps satisfy

$$
\lVert \mathbf{x}_{k+1} - \mathbf{x}^{\ast}\rVert \le \frac{M}{2m}\, \lVert \mathbf{x}_k - \mathbf{x}^{\ast}\rVert^2
$$

**Theorem 5 (BFGS positive-definiteness).** If $H_k \succ 0$ and the curvature condition $\mathbf{s}_k^T \mathbf{y}_k \gt 0$ holds, the BFGS update produces $H_{k+1} \succ 0$. Under a Wolfe line search the curvature condition holds automatically at every step.

## 3. Step-by-Step Mathematical Proofs & Derivations

### Proof 1: Armijo steps exist and backtracking terminates (Theorem 1)

Let $g = \nabla f(\mathbf{x}_k)^T \mathbf{d}_k \lt 0$ and $\varphi(\alpha) = f(\mathbf{x}_k + \alpha \mathbf{d}_k)$. Differentiability gives the expansion

$$
\varphi(\alpha) = \varphi(0) + \alpha g + o(\alpha)
$$

The Armijo requirement is $\varphi(\alpha) \le \varphi(0) + c_1 \alpha g$, equivalently

$$
\alpha (1 - c_1)\, g + o(\alpha) \le 0
$$

Since $(1 - c_1) g \lt 0$, dividing by $\alpha \gt 0$ yields $(1 - c_1) g + o(\alpha)/\alpha \le 0$, which holds for all $\alpha$ small enough that $\lvert o(\alpha)/\alpha\rvert \le (1 - c_1) \lvert g\rvert$. Hence an entire interval $(0, \bar{\alpha}]$ of Armijo steps exists.

Backtracking tries the geometric sequence $\alpha_0 \rho^m$, which enters $(0, \bar{\alpha}]$ after at most $m^{\ast} = \lceil \log(\bar{\alpha}/\alpha_0)/\log\rho \rceil$ contractions, so it terminates with an accepted step of size at least $\rho\, \bar{\alpha}$ (or $\alpha_0$ itself).

$$
\boxed{\text{descent direction} + c_1 \in (0,1) \implies \text{Armijo interval } (0, \bar{\alpha}] \text{ exists; backtracking halts}}
$$

### Proof 2: Wolfe steps exist (Theorem 2)

Define the Armijo gap $\psi(\alpha) = f(\mathbf{x}_k + \alpha \mathbf{d}_k) - f(\mathbf{x}_k) - c_1 \alpha g$ with $g = \nabla f(\mathbf{x}_k)^T \mathbf{d}_k \lt 0$.

**Step 1 — a first crossing exists.** $\psi(0) = 0$ and $\psi'(0) = (1 - c_1) g \lt 0$, so $\psi \lt 0$ for small $\alpha$. Since $f$ is bounded below along the ray while the line $f(\mathbf{x}_k) + c_1 \alpha g$ decreases without bound, $\psi(\alpha) \to +\infty$. By continuity there is a smallest $\alpha^{\prime} \gt 0$ with $\psi(\alpha^{\prime}) = 0$; on $(0, \alpha^{\prime})$ the Armijo condition holds strictly.

**Step 2 — mean value theorem.** Applied to $f$ on $[0, \alpha^{\prime}]$, there exists $\alpha^{\prime\prime} \in (0, \alpha^{\prime})$ with

$$
f(\mathbf{x}_k + \alpha^{\prime} \mathbf{d}_k) - f(\mathbf{x}_k) = \alpha^{\prime}\, \nabla f(\mathbf{x}_k + \alpha^{\prime\prime} \mathbf{d}_k)^T \mathbf{d}_k
$$

**Step 3 — the slope has risen.** The left side equals $c_1 \alpha^{\prime} g$ by definition of $\alpha^{\prime}$, so

$$
\nabla f(\mathbf{x}_k + \alpha^{\prime\prime} \mathbf{d}_k)^T \mathbf{d}_k = c_1 g \gt c_2 g
$$

using $c_2 \gt c_1$ and $g \lt 0$. Thus $\alpha^{\prime\prime}$ satisfies the curvature condition, and by Step 1 it also satisfies the Armijo condition. By continuity of $\nabla f$, a whole neighborhood of $\alpha^{\prime\prime}$ does too.

$$
\boxed{0 \lt c_1 \lt c_2 \lt 1 \text{ and } f \text{ bounded below on the ray} \implies \text{Wolfe steps form nonempty open intervals}}
$$

### Proof 3: Zoutendijk's theorem and global convergence (Theorem 3)

**Step 1 — curvature condition bounds the step from below.** Subtracting $g_k = \nabla f(\mathbf{x}_k)^T \mathbf{d}_k$ from the curvature condition $\nabla f(\mathbf{x}_{k+1})^T \mathbf{d}_k \ge c_2 g_k$:

$$
(\nabla f(\mathbf{x}_{k+1}) - \nabla f(\mathbf{x}_k))^T \mathbf{d}_k \ge (c_2 - 1)\, g_k \gt 0
$$

By the Cauchy-Schwarz inequality and $L$-Lipschitz continuity of the gradient, the left side is at most $L \alpha_k \lVert \mathbf{d}_k\rVert^2$, so

$$
\alpha_k \ge \frac{(c_2 - 1)\, g_k}{L\, \lVert \mathbf{d}_k\rVert^2} = \frac{(1 - c_2)\, \lvert g_k\rvert}{L\, \lVert \mathbf{d}_k\rVert^2}
$$

**Step 2 — Armijo converts the step bound into decrease.** The Armijo condition gives $f(\mathbf{x}_{k+1}) \le f(\mathbf{x}_k) - c_1 \alpha_k \lvert g_k\rvert$, and inserting the bound from Step 1:

$$
f(\mathbf{x}_{k+1}) \le f(\mathbf{x}_k) - \frac{c_1 (1 - c_2)}{L} \cdot \frac{g_k^2}{\lVert \mathbf{d}_k\rVert^2} = f(\mathbf{x}_k) - c\, \cos^2\theta_k\, \lVert \nabla f(\mathbf{x}_k)\rVert^2
$$

with $c = c_1(1 - c_2)/L$, using $\cos\theta_k = \lvert g_k\rvert / (\lVert \nabla f(\mathbf{x}_k)\rVert \lVert \mathbf{d}_k\rVert)$.

**Step 3 — telescope.** Summing over $k = 0, \dots, K$ and using boundedness below by $f_{\inf}$:

$$
c \sum_{k=0}^{K} \cos^2\theta_k\, \lVert \nabla f(\mathbf{x}_k)\rVert^2 \le f(\mathbf{x}_0) - f_{\inf} \lt \infty
$$

Letting $K \to \infty$ proves the Zoutendijk series converges. If $\cos\theta_k \ge \delta \gt 0$ for all $k$, the terms $\delta^2 \lVert \nabla f(\mathbf{x}_k)\rVert^2$ are summable, forcing $\lVert \nabla f(\mathbf{x}_k)\rVert \to 0$.

$$
\boxed{\sum_k \cos^2\theta_k\, \lVert \nabla f(\mathbf{x}_k)\rVert^2 \lt \infty \implies \nabla f(\mathbf{x}_k) \to \mathbf{0} \text{ whenever } \cos\theta_k \ge \delta \gt 0}
$$

### Proof 4: Newton's method converges quadratically (Theorem 4)

Write $\mathbf{e}_k = \mathbf{x}_k - \mathbf{x}^{\ast}$, $H_k = \nabla^2 f(\mathbf{x}_k)$. The full Newton step gives

$$
\mathbf{e}_{k+1} = \mathbf{e}_k - H_k^{-1} \nabla f(\mathbf{x}_k) = H_k^{-1} \left( H_k \mathbf{e}_k - \nabla f(\mathbf{x}_k) \right)
$$

**Step 1 — Taylor with integral remainder.** Since $\nabla f(\mathbf{x}^{\ast}) = \mathbf{0}$:

$$
\nabla f(\mathbf{x}_k) = \int_0^1 \nabla^2 f(\mathbf{x}^{\ast} + t\, \mathbf{e}_k)\, \mathbf{e}_k \, dt
$$

Therefore

$$
H_k \mathbf{e}_k - \nabla f(\mathbf{x}_k) = \int_0^1 \left[ \nabla^2 f(\mathbf{x}_k) - \nabla^2 f(\mathbf{x}^{\ast} + t\, \mathbf{e}_k) \right] \mathbf{e}_k \, dt
$$

**Step 2 — Lipschitz bound on the integrand.** The two Hessian arguments differ by $(1 - t)\, \mathbf{e}_k$, so $M$-Lipschitz continuity gives

$$
\lVert H_k \mathbf{e}_k - \nabla f(\mathbf{x}_k)\rVert \le \int_0^1 M (1 - t)\, \lVert \mathbf{e}_k\rVert^2 \, dt = \frac{M}{2}\, \lVert \mathbf{e}_k\rVert^2
$$

**Step 3 — bound the inverse.** Near $\mathbf{x}^{\ast}$, continuity keeps $H_k \succeq \frac{m}{2} I$ (say, for $\lVert \mathbf{e}_k\rVert \le m/(2M)$ using the Lipschitz property), hence $\lVert H_k^{-1}\rVert \le 2/m$. Combining:

$$
\lVert \mathbf{e}_{k+1}\rVert \le \frac{2}{m} \cdot \frac{M}{2}\, \lVert \mathbf{e}_k\rVert^2 = \frac{M}{m}\, \lVert \mathbf{e}_k\rVert^2
$$

(For the sharper region where $H_k \succeq m I$, the constant improves to $M/(2m)$.) Whenever $\lVert \mathbf{e}_0\rVert \lt m/M$, the sequence contracts and the error is squared each iteration — the number of correct digits doubles per step.

$$
\boxed{\lVert \mathbf{x}_{k+1} - \mathbf{x}^{\ast}\rVert \le \frac{M}{2m}\, \lVert \mathbf{x}_k - \mathbf{x}^{\ast}\rVert^2 \quad \text{near } \mathbf{x}^{\ast}}
$$

### Proof 5: Newton's method is affine invariant

Let $A$ be an invertible matrix and define the re-parameterized objective $\tilde{f}(\mathbf{y}) = f(A\mathbf{y})$. The chain rule gives

$$
\nabla \tilde{f}(\mathbf{y}) = A^T \nabla f(A\mathbf{y}), \qquad \nabla^2 \tilde{f}(\mathbf{y}) = A^T \nabla^2 f(A\mathbf{y})\, A
$$

The Newton step in the $\mathbf{y}$-coordinates is

$$
\mathbf{p}^{\,\mathbf{y}} = -\left[ A^T \nabla^2 f(A\mathbf{y})\, A \right]^{-1} A^T \nabla f(A\mathbf{y}) = -A^{-1} \left[ \nabla^2 f(A\mathbf{y}) \right]^{-1} \nabla f(A\mathbf{y}) = A^{-1} \mathbf{p}^{\,\mathbf{x}}
$$

so mapping the update back through $\mathbf{x} = A\mathbf{y}$ reproduces exactly the Newton iterates of the original problem: $A(\mathbf{y}_k + \mathbf{p}^{\,\mathbf{y}}) = \mathbf{x}_k + \mathbf{p}^{\,\mathbf{x}}$. Newton's method is blind to linear changes of coordinates — and therefore to the condition number that cripples gradient descent. Gradient descent lacks this property because $-\nabla \tilde{f} = -A^T \nabla f$ transforms with $A^T$ rather than $A^{-1}$.

$$
\boxed{\text{Newton iterates commute with invertible affine reparameterizations } \mathbf{x} = A\mathbf{y} + \mathbf{b}}
$$

### Proof 6: BFGS preserves positive definiteness (Theorem 5)

The inverse-Hessian form of the BFGS update, with $\rho_k = 1/(\mathbf{y}_k^T \mathbf{s}_k) \gt 0$ by the curvature condition, is

$$
H_{k+1} = \left( I - \rho_k\, \mathbf{s}_k \mathbf{y}_k^T \right) H_k \left( I - \rho_k\, \mathbf{y}_k \mathbf{s}_k^T \right) + \rho_k\, \mathbf{s}_k \mathbf{s}_k^T
$$

Take any $\mathbf{z} \neq \mathbf{0}$ and set $\mathbf{w} = \mathbf{z} - \rho_k (\mathbf{s}_k^T \mathbf{z})\, \mathbf{y}_k$. Then

$$
\mathbf{z}^T H_{k+1} \mathbf{z} = \mathbf{w}^T H_k \mathbf{w} + \rho_k\, (\mathbf{s}_k^T \mathbf{z})^2
$$

Both terms are nonnegative ($H_k \succ 0$, $\rho_k \gt 0$). Suppose the sum is zero. Then $\mathbf{s}_k^T \mathbf{z} = 0$ and $\mathbf{w}^T H_k \mathbf{w} = 0$; the first forces $\mathbf{w} = \mathbf{z}$, and the second, by positive definiteness of $H_k$, forces $\mathbf{z} = \mathbf{0}$ — a contradiction. Hence $\mathbf{z}^T H_{k+1} \mathbf{z} \gt 0$ for all $\mathbf{z} \neq \mathbf{0}$.

Finally, the Wolfe curvature condition guarantees $\mathbf{y}_k^T \mathbf{s}_k \gt 0$ automatically: it states $\nabla f(\mathbf{x}_{k+1})^T \mathbf{d}_k \ge c_2 \nabla f(\mathbf{x}_k)^T \mathbf{d}_k$, and subtracting $\nabla f(\mathbf{x}_k)^T \mathbf{d}_k$ from both sides gives $\mathbf{y}_k^T \mathbf{d}_k \ge (c_2 - 1) \nabla f(\mathbf{x}_k)^T \mathbf{d}_k \gt 0$, and $\mathbf{s}_k = \alpha_k \mathbf{d}_k$ with $\alpha_k \gt 0$.

$$
\boxed{H_k \succ 0 \text{ and } \mathbf{s}_k^T \mathbf{y}_k \gt 0 \implies H_{k+1} \succ 0 \quad \text{(guaranteed under Wolfe line search)}}
$$

## 4. Computational & Algorithmic Insights

### The practical line-search recipe

Production optimizers use backtracking with $c_1 = 10^{-4}$: the sufficient-decrease bar is set extremely low, accepting almost any genuine decrease, while the initial trial $\alpha_0 = 1$ ensures that Newton and quasi-Newton methods can take their natural unit step the moment it becomes acceptable — preserving fast local convergence. Typical contraction is $\rho \in [0.1, 0.5]$, often with polynomial interpolation of $\varphi$ instead of blind halving. For BFGS the *strong* Wolfe conditions (with $\lvert \varphi'(\alpha)\rvert \le c_2 \lvert \varphi'(0)\rvert$, $c_2 = 0.9$) are preferred because they also guarantee the curvature condition of Proof 6.

### Cost anatomy per iteration

| Method | Direction cost | Memory | Local rate |
|---|---|---|---|
| Gradient descent | $O(n)$ | $O(n)$ | linear, factor $1 - \mu/L$ |
| Newton | $O(n^3)$ solve + $O(n^2)$ Hessian | $O(n^2)$ | quadratic |
| BFGS | $O(n^2)$ update | $O(n^2)$ | superlinear |
| L-BFGS ($m$ pairs) | $O(mn)$ two-loop | $O(mn)$ | linear-to-superlinear |

### The L-BFGS two-loop recursion

L-BFGS never forms $H_k$: it stores the last $m$ pairs $(\mathbf{s}_i, \mathbf{y}_i)$ and computes $H_k \nabla f(\mathbf{x}_k)$ by:

1. **First loop (newest to oldest):** $q \leftarrow \nabla f(\mathbf{x}_k)$; for each stored pair compute $a_i = \rho_i\, \mathbf{s}_i^T q$ and update $q \leftarrow q - a_i\, \mathbf{y}_i$.
2. **Scaling:** $r \leftarrow \gamma_k q$ with $\gamma_k = \mathbf{s}_{k-1}^T \mathbf{y}_{k-1} / \mathbf{y}_{k-1}^T \mathbf{y}_{k-1}$, a spectral estimate of the inverse-Hessian scale.
3. **Second loop (oldest to newest):** $b_i = \rho_i\, \mathbf{y}_i^T r$; update $r \leftarrow r + (a_i - b_i)\, \mathbf{s}_i$.

The output $r = H_k \nabla f(\mathbf{x}_k)$ costs $4mn$ multiplications. With $m$ between 5 and 20, this scales to millions of variables and is the default solver in `scipy.optimize.minimize(method="L-BFGS-B")` and classical ML libraries.

### Globalization safeguards

Raw Newton can fail far from the solution: the Hessian may be indefinite (direction points uphill) or singular. Standard fixes: (i) **damped Newton** — always line-search the Newton direction; (ii) **Hessian modification** — factor $H + \tau I$ with the smallest $\tau \ge 0$ making Cholesky succeed; (iii) **trust regions** — minimize the quadratic model inside a ball $\lVert \mathbf{p}\rVert \le \Delta_k$ whose radius adapts to model quality; (iv) **fallback** — if the modified direction fails a descent test, revert to $-\nabla f$, which keeps $\cos\theta_k$ bounded away from zero and re-enters Zoutendijk's hypothesis.

## 5. Real-World Physics & AI/ML Applications

### Newton = Iteratively Reweighted Least Squares in statistics

For logistic regression with design matrix $X$ and labels $y_i \in \{0, 1\}$, the negative log-likelihood has

$$
\nabla \ell(\mathbf{w}) = X^T (\boldsymbol{\pi} - \mathbf{y}), \qquad \nabla^2 \ell(\mathbf{w}) = X^T S X, \quad S = \operatorname{diag}\left( \pi_i (1 - \pi_i) \right)
$$

where $\pi_i = \sigma(\mathbf{x}_i^T \mathbf{w})$. The Newton step solves $X^T S X\, \Delta\mathbf{w} = X^T(\mathbf{y} - \boldsymbol{\pi})$ — a *weighted least-squares problem* re-solved each iteration with updated weights $S$. This is the IRLS algorithm inside every classical GLM package, converging in a handful of iterations thanks to quadratic convergence.

### Physics: equilibrium and structure relaxation

Finding stable molecular geometries or elastic equilibria means minimizing an energy $E(\mathbf{q})$ over thousands to millions of coordinates. Quantum-chemistry and molecular-mechanics codes overwhelmingly use BFGS/L-BFGS: exact Hessians of the energy are expensive, but gradients (forces) are cheap, and the secant machinery converts force histories into curvature — exactly the quasi-Newton premise. Convergence is declared when the maximum force component drops below a threshold, i.e. the FONC of Topic 02.

### Why deep learning does not use these methods directly — and where they return

Second-order methods struggle in deep learning: the Hessian is astronomically large, mini-batch noise breaks the secant equation ($\mathbf{y}_k$ contains gradient noise, violating $\mathbf{s}_k^T \mathbf{y}_k \gt 0$), and the loss is non-convex. Yet curvature ideas persist:

- **K-FAC and Shampoo** approximate the Fisher/Hessian with Kronecker or block structure — structured quasi-Newton methods.
- **Adam** (Topic 08) is a diagonal curvature estimate: element-wise step scaling by $1/\sqrt{\hat{v}_k}$ mimics a diagonal inverse Hessian.
- **L-BFGS remains the tool of choice** for full-batch problems in scientific ML: physics-informed neural networks, style transfer, and small-data fitting almost always finish training with L-BFGS after an Adam warm start.

## 6. Canonical Literature Mapping & References

| Result in this notebook | Canonical source |
|---|---|
| Armijo condition and backtracking (Proof 1) | Armijo (1966); Nocedal & Wright Section 3.1 |
| Wolfe conditions and existence (Proof 2) | Wolfe (1969); Nocedal & Wright Lemma 3.1 |
| Zoutendijk's theorem (Proof 3) | Zoutendijk (1970); Nocedal & Wright Theorem 3.2 |
| Newton quadratic convergence (Proof 4) | Nocedal & Wright Theorem 3.5; Boyd & Vandenberghe Section 9.5.3 |
| Affine invariance (Proof 5) | Boyd & Vandenberghe Section 9.5.1 |
| BFGS SPD preservation (Proof 6) | Nocedal & Wright Section 6.1; Dennis & Schnabel Ch. 9 |
| L-BFGS two-loop recursion | Liu & Nocedal (1989); Nocedal & Wright Algorithm 7.4 |

1. **Nocedal, J., & Wright, S. J.** (2006). *Numerical Optimization* (2nd ed.). Springer. Chapters 3, 6, 7.
2. **Boyd, S., & Vandenberghe, L.** (2004). *Convex Optimization*. Cambridge University Press. Chapter 9.
3. **Dennis, J. E., & Schnabel, R. B.** (1996). *Numerical Methods for Unconstrained Optimization and Nonlinear Equations*. SIAM. Chapters 6-9.
4. **Bertsekas, D. P.** (2016). *Nonlinear Programming* (3rd ed.). Athena Scientific. Chapter 1.
5. **Liu, D. C., & Nocedal, J.** (1989). On the Limited Memory BFGS Method for Large Scale Optimization. *Mathematical Programming*, 45, 503-528.
6. **Armijo, L.** (1966). Pacific J. Math. 16(1); **Wolfe, P.** (1969). SIAM Review 11(2); **Zoutendijk, G.** (1970). in *Integer and Nonlinear Programming*, North-Holland.